# Predição da próxima palavra com uma LSTM

Este notebook apresenta um exemplo **simples e didático** de previsão da próxima palavra usando uma rede LSTM (Long Short-Term Memory) com TensorFlow/Keras.

**Ideia:** fornecemos algumas palavras como contexto (por exemplo, `o gato`) e o modelo aprende a prever a palavra seguinte (por exemplo, `dorme`).

> **Importante:** o conjunto de frases é pequeno e criado apenas para demonstração. O modelo pode memorizar padrões e não terá a capacidade de um modelo de linguagem treinado em grandes volumes de texto.

## 1. Instalação

Se necessário, descomente e execute a célula abaixo. O TensorFlow pode demorar um pouco para instalar dependendo do ambiente.

In [11]:
# Se o TensorFlow não estiver instalado, descomente:
# %pip install tensorflow

## 2. Importar bibliotecas e definir frases

Usaremos frases em português. Para simplificar, a tokenização será feita por palavras e a pontuação será separada previamente de forma básica.

In [12]:
import re
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

frases = [
    "o gato dorme no sofá",
    "o gato gosta de leite",
    "o gato brinca no jardim",
    "o cachorro dorme no quintal",
    "o cachorro gosta de brincar",
    "o cachorro corre no parque",
    "a menina gosta de música",
    "a menina lê um livro",
    "a menina brinca no jardim",
    "o menino lê um livro",
    "o menino gosta de futebol",
    "o menino corre no parque",
    "o sol brilha no céu",
    "a lua aparece no céu",
    "o pássaro canta na árvore",
    "o pássaro voa no céu",
    "a professora lê um livro",
    "a professora gosta de ensinar",
    "o estudante lê um livro",
    "o estudante gosta de aprender",
    "o gato corre no jardim",
    "o cachorro brinca no parque",
    "a menina canta uma música",
    "o menino canta uma música",
    "o café está muito quente",
    "o chá está muito quente",
    "a água está muito fria",
    "o dia está muito bonito",
    "a noite está muito bonita",
    "o gato está muito quieto",

    # Animais
    "o gato bebe água",
    "o gato sobe na árvore",
    "o cachorro protege a casa",
    "o cachorro bebe água",
    "o pássaro constrói um ninho",
    "o peixe nada no rio",
    "a tartaruga anda devagar",
    "o cavalo corre no campo",
    "a borboleta pousa na flor",
    "o coelho come uma cenoura",

    # Pessoas e atividades
    "a menina estuda matemática",
    "o menino aprende português",
    "a professora explica a matéria",
    "o estudante faz a tarefa",
    "o médico trabalha no hospital",
    "a enfermeira cuida dos pacientes",
    "o cozinheiro prepara o jantar",
    "o agricultor planta milho",
    "o músico toca violão",
    "a criança desenha uma casa",

    # Casa e objetos
    "a mãe prepara o almoço",
    "o pai lê o jornal",
    "a família assiste televisão",
    "o bebê dorme no berço",
    "a porta está aberta",
    "a janela está fechada",
    "o relógio marca as horas",
    "o telefone toca muito alto",
    "a lâmpada ilumina o quarto",
    "o computador está ligado",

    # Natureza e clima
    "a chuva molha a rua",
    "o vento balança as árvores",
    "o sol aquece a terra",
    "as nuvens cobrem o céu",
    "o rio atravessa a cidade",
    "as flores crescem no jardim",
    "a grama está muito verde",
    "a neve cobre a montanha",
    "o mar está tranquilo",
    "a tempestade chega à noite",

    # Comida e bebidas
    "o café está quente",
    "o leite está frio",
    "a menina come uma maçã",
    "o menino bebe suco de laranja",
    "a família prepara um bolo",
    "o pão está fresco",
    "a sopa está muito quente",
    "o cozinheiro corta os legumes",
    "o arroz acompanha o feijão",
    "a água mata a sede",

    # Tecnologia e ciência
    "o computador processa os dados",
    "o celular recebe uma mensagem",
    "o programa executa uma tarefa",
    "a rede transmite informações",
    "o sensor mede a temperatura",
    "o robô aprende uma tarefa",
    "a câmera captura imagens",
    "o satélite observa a Terra",
    "a inteligência artificial reconhece padrões",
    "o pesquisador analisa os resultados",
]

print(f"Número de frases: {len(frases)}")
print("Exemplos:")
for frase in frases[:5]:
    print(" •", frase)

Número de frases: 90
Exemplos:
 • o gato dorme no sofá
 • o gato gosta de leite
 • o gato brinca no jardim
 • o cachorro dorme no quintal
 • o cachorro gosta de brincar


## 3. Transformar palavras em números

Uma rede neural não recebe palavras diretamente. O `Tokenizer` cria um vocabulário e associa um índice inteiro a cada palavra. Em seguida, geramos exemplos de treino com todos os prefixos de cada frase.

Exemplo para `o gato dorme no sofá`:
- `o` → `gato`
- `o gato` → `dorme`
- `o gato dorme` → `no`
- `o gato dorme no` → `sofá`

Todos os exemplos são completados até o mesmo comprimento usando padding.

In [13]:
tokenizer = Tokenizer(oov_token="<desconhecida>")
tokenizer.fit_on_texts(frases)
vocab_size = len(tokenizer.word_index) + 1  # índice 0 reservado para padding

sequencias = []
for frase in frases:
    ids = tokenizer.texts_to_sequences([frase])[0]
    for i in range(1, len(ids)):
        sequencias.append(ids[:i + 1])

max_len = max(len(seq) for seq in sequencias)
sequencias_pad = pad_sequences(sequencias, maxlen=max_len, padding="pre")

X = sequencias_pad[:, :-1]  # contexto: todas as palavras menos a última
y = sequencias_pad[:, -1]   # alvo: próxima palavra

print("Tamanho do vocabulário:", vocab_size)
print("Número de exemplos de treino:", len(X))
print("Comprimento máximo do contexto:", X.shape[1])

# Inspecione alguns exemplos em formato legível
indice_para_palavra = {idx: palavra for palavra, idx in tokenizer.word_index.items()}
for x_ex, y_ex in zip(X[:5], y[:5]):
    contexto = [indice_para_palavra.get(int(i), "<PAD>") for i in x_ex if i != 0]
    alvo = indice_para_palavra.get(int(y_ex), "?")
    print(f"{' '.join(contexto):25s} -> {alvo}")

Tamanho do vocabulário: 188
Número de exemplos de treino: 344
Comprimento máximo do contexto: 5
o                         -> gato
o gato                    -> dorme
o gato dorme              -> no
o gato dorme no           -> sofá
o                         -> gato


## 4. Criar e treinar a LSTM

A camada `Embedding` aprende uma representação vetorial para cada palavra. A LSTM processa a sequência de palavras na ordem em que aparece e a camada `Dense` produz uma pontuação para cada palavra do vocabulário. Com `softmax`, essas pontuações podem ser interpretadas como probabilidades.

In [14]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=32, mask_zero=True),
    LSTM(64),
    Dense(vocab_size, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [15]:
history = model.fit(
    X, y,
    epochs=250,
    batch_size=16,
    verbose=0
)

print("Treinamento concluído.")
print(f"Acurácia no próprio conjunto de treino: {history.history['accuracy'][-1]:.3f}")
print("Nota: essa acurácia é calculada nos dados usados para treinar; não mede generalização.")

Treinamento concluído.
Acurácia no próprio conjunto de treino: 0.651
Nota: essa acurácia é calculada nos dados usados para treinar; não mede generalização.


## 5. Predizer a próxima palavra

A função abaixo recebe um contexto de palavras, transforma-o em números, consulta o modelo e retorna as palavras com maior probabilidade.

In [16]:
def prever_proxima_palavra(contexto, top_k=5):
    ids = tokenizer.texts_to_sequences([contexto])[0]
    if not ids:
        raise ValueError("Digite ao menos uma palavra conhecida pelo tokenizador.")

    entrada = pad_sequences([ids], maxlen=max_len - 1, padding="pre")
    probabilidades = model.predict(entrada, verbose=0)[0]

    # Não sugerir o índice 0 (padding)
    indices = np.argsort(probabilidades[1:])[::-1][:top_k] + 1
    return [
        (indice_para_palavra.get(int(i), "?"), float(probabilidades[i]))
        for i in indices
    ]

contextos_teste = [
    "o gato",
    "o cachorro",
    "a menina",
    "o menino lê",
    "o pássaro"
]

for contexto in contextos_teste:
    print(f"Contexto: {contexto!r}")
    for palavra, prob in prever_proxima_palavra(contexto, top_k=3):
        print(f"   {palavra:15s} probabilidade = {prob:.3f}")
    print()

Contexto: 'o gato'
   está            probabilidade = 0.148
   dorme           probabilidade = 0.146
   corre           probabilidade = 0.142

Contexto: 'o cachorro'
   corre           probabilidade = 0.172
   gosta           probabilidade = 0.171
   dorme           probabilidade = 0.164

Contexto: 'a menina'
   gosta           probabilidade = 0.177
   come            probabilidade = 0.164
   estuda          probabilidade = 0.159

Contexto: 'o menino lê'
   um              probabilidade = 0.995
   gosta           probabilidade = 0.001
   lê              probabilidade = 0.001

Contexto: 'o pássaro'
   constrói        probabilidade = 0.329
   canta           probabilidade = 0.314
   voa             probabilidade = 0.291



## 6. Experimente seus próprios contextos

Edite o texto na célula seguinte e execute-a. Como o vocabulário é pequeno, palavras desconhecidas podem ser ignoradas pelo tokenizador. A previsão é apenas uma demonstração do mecanismo, não uma resposta garantidamente correta.

In [17]:
contexto = "o gato dorme"
print("Contexto:", contexto)
print("Próximas palavras mais prováveis:")
for palavra, prob in prever_proxima_palavra(contexto, top_k=5):
    print(f"  {palavra:15s} {prob:.3f}")

Contexto: o gato dorme
Próximas palavras mais prováveis:
  no              0.993
  muito           0.002
  uma             0.002
  água            0.001
  gosta           0.000


In [18]:
contexto = "o gato dorme no"
print("Contexto:", contexto)
print("Próximas palavras mais prováveis:")
for palavra, prob in prever_proxima_palavra(contexto, top_k=5):
    print(f"  {palavra:15s} {prob:.3f}")

Contexto: o gato dorme no
Próximas palavras mais prováveis:
  sofá            0.923
  quintal         0.035
  jardim          0.017
  berço           0.010
  campo           0.003


In [19]:
contexto = "o gato dorme no sofá"
print("Contexto:", contexto)
print("Próximas palavras mais prováveis:")
for palavra, prob in prever_proxima_palavra(contexto, top_k=5):
    print(f"  {palavra:15s} {prob:.3f}")

Contexto: o gato dorme no sofá
Próximas palavras mais prováveis:
  bonito          0.336
  berço           0.097
  sofá            0.084
  quieto          0.077
  laranja         0.060


## 7. O que observar

- **Contexto:** as palavras anteriores são a entrada do modelo.
- **Alvo:** a palavra seguinte é a resposta esperada durante o treino.
- **LSTM:** mantém um estado interno que ajuda a representar dependências na sequência.
- **Softmax:** gera uma distribuição de probabilidades sobre as palavras do vocabulário.
- **Limitação:** este exemplo usa poucas frases e avalia as previsões nos mesmos padrões usados no treino. Para medir desempenho real, seria necessário um conjunto de teste separado e um corpus maior.

### Ideias para continuar
1. Acrescente dezenas ou centenas de frases variadas.
2. Separe frases de treino e teste antes de criar os exemplos de prefixos.
3. Compare LSTM com uma baseline simples, como prever sempre a palavra mais frequente.
4. Experimente prever mais de uma palavra, alimentando cada previsão de volta como contexto.